#📓 RAG Framework Evaluation and Upgrade


<img src="https://i.imgur.com/fTICSCN.png">

In this exercise, you will have the opportunity to evaluate the various RAG components you have built earlier. In the previous Day 1 session, we only performed evaluation using the LLM. However, this time, we will utilize various evaluation methods commonly used in the NLP field.


### I. Evaluate RAG  
### II. Agentic Loop: Letting the System Decide Whether to Look Again
### III. Upgrade KG Query Stage with MCP
  
Okay. Now we know what we have to do for this final section.  
However, we need to know additional evaluation metric for CRAG dataset:

`Exact Accuracy`  
`Accuracy`  
`Hallucination`   
`Missing`  



## 0. New evaluation metrics for CRAG dataset

In most cases, datasets designed for specific tasks are presented along with **evaluation metrics** that can be used for performance evaluation. Similarly, the CRAG dataset provides evaluation metrics that should be used when measuring the performance of LLMs on this dataset.  

Therefore, before proceeding with the evaluation, let’s first check which evaluation methods the creators of the CRAG dataset intended to use. Specifically, we will examine the four answer classification criteria they proposed, understand how these are evaluated, and clarify what each criterion means.  

<br/>

We follow the steps below:  

#### 1. What is the new evaluation metrics for CRAG dataset?
#### 2. How to evalute RAG following new evaluation metrics?


### 1. What is the new evaluation metrics for CRAG dataset?
  
The creators of the CRAG dataset evaluated RAG based on the following four elements:

<img src="https://i.imgur.com/0hxmPdi.png">

Simply put, they classified responses that were identical to their predefined answers as the most ideal case. Responses with similar meanings but containing minor errors were classified as the next most ideal case.

The important point is that, under the CRAG dataset’s evaluation criteria, everything else is not simply classified as “incorrect.” Instead, the creators expect the LLM to admit when it does not know the answer. Incorrect answers are those containing errors, and the more these answers occur, the worse the model’s performance is considered. However, answers classified as Missing (indicating no answer) do not negatively or positively impact the model’s performance.

By understanding these four classification criteria, you will gain valuable insights when analyzing evaluation results on the CRAG dataset.


### 2. How to evalute RAG following new evaluation metrics?
  
Since the evaluation criteria mentioned above cannot be measured automatically, the evaluation must be conducted using an LLM. This can be done using methods similar to Trulens. However, as the evaluation results can vary depending on the prompt used, we will use the default prompt provided by the creators of the CRAG dataset.

The evaluation prompt is as follows. Based on its content, we need to provide the LLM with the `question`, `model prediction`, and `ground truth answers`. The LLM will then generate a response by performing the evaluation according to the instructions.

To help us understand this with a simple example, please install and import the following library:

In [ ]:
import os
os.environ.pop("PYTHONPATH", None)

!pip install openai==1.81.0 \
    llama-index llama-index-embeddings-huggingface==0.4.0 llama-index-tools-mcp \
    packaging>=24.0 langchain nltk>=3.8.1 \
    streamlit==1.35.0 watchdog kubernetes==26.1.0 \
    blingfire beautifulsoup4 sentence-transformers ray scikit-learn \
    tqdm tiktoken transformers matplotlib numpy --quiet

In [ ]:
### YOUR CODE HERE ###

INSTRUCTIONS = """
# Task:
You are given a Question, a model Prediction, and a list of Ground Truth answers, judge whether the model Prediction matches any answer from the list of Ground Truth answers. Follow the instructions step by step to make a judgement.
1. If the model prediction matches any provided answers from the Ground Truth Answer list, "Accuracy" should be "True"; otherwise, "Accuracy" should be "False".
2. If the model prediction says that it couldn't answer the question or it doesn't have enough information, "Accuracy" should always be "False".
3. If the Ground Truth is "invalid question", "Accuracy" is "True" only if the model prediction is exactly "invalid question".
# Output:
Respond with only a single JSON string with an "Accuracy" field which is "True" or "False".
"""

import sys

You can drag-and-drop the import code file into the workspace. This will allow you to import the necessary functions from that file for this practice. However, please ensure that the file is in the same folder as the currently running code for the import to succeed.  


In [ ]:
### YOUR CODE HERE ###

import os
os.environ["OPENAI_API_KEY"] = "sk-..." #Insert your openai api key

import openai
import json
import random
import bz2
from tqdm import tqdm
from import_function import LlamaIndexRetriever, Reader, KGQueryEngine

Now, to proceed with the evaluation, let’s make the dataset accessible. Run the code below. Depending on your computer environment, this may take a little time.  

In [ ]:
### YOUR CODE HERE ###

file_path = '/path/to/CRAG dataset/crag_task_1_dev_v4_release.jsonl.bz2'

dataset = []

with bz2.open(file_path, 'rt') as file:
    for line in file:
        try:
            data = json.loads(line.strip())
            dataset.append(data)
            if len(dataset) > 500:
              break
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

Please note that, depending on your computer environment, this process might take some time.  

<img src="https://i.gifer.com/B6Qs.gif" width="150">

Thank you for your understanding.


Next, to obtain the model prediction by asking the LLM a question, we will use the following simple code to generate a response:


In [ ]:
### YOUR CODE HERE ###

def generate_answer(user_prompt, system_prompt = "You are a helpful assistant."):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
    )

    return response.choices[0].message.content

Now, let’s proceed with the evaluation. The evaluation will be conducted using randomly selected data points.

In [ ]:
### YOUR CODE HERE ###

random_data = random.choice(dataset)

test_question = random_data['query']
test_answer = random_data['answer']
test_answer_candidate = random_data['alt_ans']
all_answers = [test_answer] + test_answer_candidate

model_prediction = generate_answer(user_prompt=test_question)

In [ ]:
### YOUR CODE HERE ###

context_template = f"""Question: {test_question}
List of Ground Truth answers: {all_answers}
Model Prediction: {model_prediction}
"""

print(context_template)

evaluation_result = generate_answer(user_prompt=context_template, system_prompt=INSTRUCTIONS)

print(evaluation_result)

Once the LLM evaluation results are generated, we need a function to extract the relevant results. Using the parser below, we can extract the classification results effectively:


In [ ]:
### YOUR CODE HERE ###

def parse_response(response):
    try:
        response = response.lower()
        model_resp = json.loads(response)
        answer = -1
        if "accuracy" in model_resp and (
            (
              model_resp["accuracy"] is True
            )
            or
            (
                isinstance(model_resp["accuracy"], str)
                and model_resp["accuracy"].lower() == "true"
            )
        ):
            answer = 1
        else:
            raise ValueError(f"Could not parse answer from response: {model_resp}")

        return answer
    except:
        return -1

test_text = """{
    "Accuracy": "True"
}"""

print(parse_response(test_text))

Here is the function that processes the LLM-generated response, converts it into JSON format, and checks for the accuracy attribute. If accuracy is true, it returns 1; otherwise, it returns -1.

This function can be used to perform the evaluation.

In [ ]:
### YOUR CODE HERE ###

def CRAG_evaluation(question, ground_truth, prediction):
  context_template = f"""Question: {question}
  List of Ground Truth answers: {ground_truth}
  Model Prediction: {prediction}
  """

  evaluation_result = generate_answer(user_prompt=context_template, system_prompt=INSTRUCTIONS)

  eval_res = parse_response(evaluation_result)

  return eval_res

## I. Evaluate RAG

So far, we have evaluated the performance of the retriever to determine which retriever can be more effective. While using the recall score as the evaluation metric makes it difficult to measure semantic similarity, we have seen that it is convenient for large-scale automatic evaluation.

In this section, we aim to evaluate the RAG system using multiple approaches. After selecting the best retriever based on the method described above, we now need to integrate it with the Reader to build a complete RAG system and verify its overall performance.
We follow the steps below:  

#### 1. Define RAG and Evaluation Metric
#### 2. Evaluate through CRAG Evaluation Method

### 1. Define RAG
Now, we will define RAG class.

In [ ]:
### YOUR CODE HERE ###

external_kg_server = "http://10.2.0.165:8000"   #need to change

class RAG:
    def __init__(self, server=None):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine(server=server)
        self.reader = Reader()

    def retrieve(self, query, search_results, topk):
        retrieved_results = self.retriever.retrieve(query, search_results, topk)

        kg_results = self.kg_query_engine.query(query)

        combined_results = [kg_results]
        combined_results.extend(retrieved_results)

        return combined_results

    def generate_response(self, query, retrieved_results):
        answer = self.reader.generate_response(query, retrieved_results)
        return answer

    def inference(self, query, search_results, topk):
        retrieved_results = self.retrieve(query, search_results, topk)
        answer = self.generate_response(query, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }

rag = RAG(server=external_kg_server)

###2. Evaluate through CRAG Evaluation Method

This time, we will evaluate the results using the evaluation metrics proposed in the CRAG dataset. As explained at the very beginning of this session, the model’s predictions are categorized into four classes for evaluation purposes.

####1.   **Perfect**: Correctly answers the question and contains no hallucination
####2.   **Acceptable**: Provide a useful answer to the question but may contain minor errors
####3.   **Missing**: The response is "I don't know", "I'm sorry I can't find ...".
####4.   **Incorrect**: The response provides wrong or irrelevant infromation to answer the question.

The model predictions categorized above are then linearized in the following manner to evaluate the final performance of the RAG system.

<img src="https://i.imgur.com/TDQ5eI4.png">

Here, let’s proceed to evaluate the validation set using the function we just defined.

We designed our Graph RAG to handle only questions within the finance domain, so we will also select test dataset questions that belong to the finance domain.

In [ ]:
### YOUR CODE HERE ###

finance_test_dataset_ids = []

for data in dataset:
    if data['domain'] == 'finance':
        finance_test_dataset_ids.append(data['interaction_id'])

    if len(finance_test_dataset_ids) >= 10:
        break

In [ ]:
### YOUR CODE HERE ###

n_miss, n_correct, n_correct_exact = 0, 0, 0

for data in tqdm(dataset):
  if data['interaction_id'] not in finance_test_dataset_ids:
    continue

  question = data['query']
  ground_truth_lowercase = str(data['answer']).strip().lower()
  web_search_results = data['search_results']

  prediction_lowercase = rag.inference(question, web_search_results, 5)['answer'].lower()

  if prediction_lowercase == ground_truth_lowercase:
      n_correct_exact += 1
      continue
  elif "i don't know" in prediction_lowercase:
      n_miss += 1
      continue

  acceptable = CRAG_evaluation(question, ground_truth_lowercase, prediction_lowercase)

  if acceptable == 1:
    n_correct += 1

n_hallucinate = (len(finance_test_dataset_ids) - n_correct_exact - n_correct - n_miss)

CRAG_score = n_correct_exact + 0.5*n_correct - n_hallucinate

print("\n\n")
print("Number of correct answers:", n_correct)
print("Number of exact correct answers:", n_correct_exact)
print("Number of missed answers:", n_miss)
print("Number of hallucinated answers:", n_hallucinate)
print("CRAG score:", CRAG_score)

## II. Agentic Loop: Letting the System Decide Whether to Look Again

What we measured in Section I was a **single-pass RAG**: retrieve once, read once, done. All the
Reader ever saw was whatever the retriever handed over. If that evidence was incomplete, or about the
wrong entity, or about the wrong point in time, the Reader still produced a confident answer — and
the CRAG score charged us **-1** for it.

The problem is not that it searched only once. The problem is that **nobody ever checked whether once
was enough**. A single-pass RAG has nowhere to ask "does this evidence actually answer the question?",
so it does exactly the same thing for a question with abundant evidence and for one with none.

Most questions do fine with a single search. That is why single-pass RAG works at all. But some do
not. Sometimes the question as written is a poor retrieval key and drags in the wrong documents;
sometimes the fact is simply not on the web and lives only in the Knowledge Graph. Those questions
need **another look**. The catch is that you cannot tell in advance which questions those are.

The idea behind an **agentic loop** is that you do not need to tell in advance. After producing an
answer, the system checks whether the evidence really supports it, and if it does not, uses the gap
as the target for the next search.

```
plan -> act (retrieve) -> read -> verify
```

If verification passes, it stops there. If it does not, the system goes around again, aiming at
whatever the verifier said was missing. Easy questions finish in one turn; only hard ones take two or
three. **Deciding how many searches to run, per question, at run time rather than in advance** — that
is all the loop adds.

Making that decision needs a few capabilities the system did not have, and the loop brings them along
together:

- **Feedback** — the verifier says what is missing, and that complaint determines the next search.
- **Routing** — each turn, choose between the web and the Knowledge Graph.
- **Query rewriting** — search for what is missing, not for the question as originally worded.
- **Abstention** — when the evidence never arrives, say so instead of inventing it.

To find out whether these actually raise the score, we build a ladder that adds one structure at a
time and run all of it on **the same evaluation set with the same CRAG metric**.

|      | System                               | Structure added                        |
|------|--------------------------------------|----------------------------------------|
| **L0** | LLM only, no retrieval             | (none)                                 |
| **L1** | Single-pass RAG (Section I)        | + retrieval                            |
| **L2** | L1 + one self-verification step    | + verification / abstention            |
| **L3** | Agentic loop with a web/KG router  | + feedback / routing / query rewriting |
| **L4** | MCP tool-calling agent (Section III) | + protocol-driven tool selection     |

Keep one thing in mind while reading the results. The CRAG score does not merely reward correct
answers — it **punishes confident wrong ones**. So much of what the loop brings in is not
`incorrect -> perfect` but `incorrect -> missing`: learning to say *"I don't know"* instead of
guessing. That is a `-1 -> 0` swing, worth as much as getting two more answers half-right.


### 1. Attaching a Meter

The code below counts how many times the system calls the LLM. For now we are only attaching the
meter; why it matters is better discussed in Section 6, once we have seen the results.

Our components do not share an OpenAI client — `Reader` and `KGQueryEngine` use the one inside
`import_function`, `generate_answer` uses the module-level `openai`, and the MCP agent in Section III
uses LlamaIndex's own. Rather than editing all of them, we wrap `chat.completions.create` itself, so
every call from every component is counted in one place.

Two notes on what falls inside the count:

- **Embedding calls are not counted.** The endpoint the retriever uses to embed chunks is priced very
  differently. Folding it into the same number would blur the comparison we make later.
- **Streamed responses report no token usage.** The MCP agent in Section III streams its events, so
  its *call count* is exact while its *token count* is a lower bound.


In [ ]:
### YOUR CODE HERE ###

from openai import OpenAI
from openai.resources.chat import completions as _oai_completions

oai_client = OpenAI()

_ORIG_SYNC_CREATE = _oai_completions.Completions.create
_ORIG_ASYNC_CREATE = _oai_completions.AsyncCompletions.create


class LLMCallCounter:
    """Context manager that counts chat-completion calls made inside its block."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.calls = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0

    def _record(self, response):
        self.calls += 1
        usage = getattr(response, "usage", None)  # absent when stream=True
        if usage is not None:
            self.prompt_tokens += getattr(usage, "prompt_tokens", 0) or 0
            self.completion_tokens += getattr(usage, "completion_tokens", 0) or 0
        return response

    def __enter__(self):
        counter = self

        def sync_create(self, *args, **kwargs):
            return counter._record(_ORIG_SYNC_CREATE(self, *args, **kwargs))

        async def async_create(self, *args, **kwargs):
            return counter._record(await _ORIG_ASYNC_CREATE(self, *args, **kwargs))

        _oai_completions.Completions.create = sync_create
        _oai_completions.AsyncCompletions.create = async_create
        return self

    def __exit__(self, *exc_info):
        _oai_completions.Completions.create = _ORIG_SYNC_CREATE
        _oai_completions.AsyncCompletions.create = _ORIG_ASYNC_CREATE
        return False


# Sanity check: one call in, one call counted.
with LLMCallCounter() as counter:
    generate_answer("Say 'ok'.")

print(f"calls={counter.calls}, "
      f"prompt_tokens={counter.prompt_tokens}, "
      f"completion_tokens={counter.completion_tokens}")


### 2. One Evaluation Set, One Metric, Five Systems

Section I evaluated on 10 finance questions. That is enough to sanity-check a single system, but not
to separate five of them: with 10 questions a two-point difference in CRAG score can easily be noise.
We widen the set to **30** here.

`evaluate_system()` below is Section I's scoring loop, generalised so that every system on the ladder
is scored in exactly the same way. Note where the meter sits: it wraps **only the inference**, never
`CRAG_evaluation`. The judge is itself an LLM call, and counting it would add the same amount to every
system while telling us nothing.

Results are cached to `agentic_loop_results.json`. Running the whole ladder takes a while, so during
class you can load the cache, or drop `N_EVAL` to 5 and watch it run.


In [ ]:
### YOUR CODE HERE ###

import time
import numpy as np

N_EVAL = 30                                  # lower this to 5 if you want to watch it run live
CACHE_PATH = "agentic_loop_results.json"

agentic_eval_ids = []
for data in dataset:
    if data["domain"] == "finance":
        agentic_eval_ids.append(data["interaction_id"])
    if len(agentic_eval_ids) >= N_EVAL:
        break

_eval_id_set = set(agentic_eval_ids)
eval_items = [d for d in dataset if d["interaction_id"] in _eval_id_set]
print(f"# of evaluation items: {len(eval_items)}")


RESULTS = {}
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        RESULTS = json.load(f)
    print(f"loaded cached results: {list(RESULTS.keys())}")


def save_results():
    with open(CACHE_PATH, "w") as f:
        json.dump(RESULTS, f, indent=2)


def grade_prediction(item, prediction):
    """Return one of the four CRAG grades: perfect / acceptable / missing / incorrect."""
    ground_truth = str(item["answer"]).strip().lower()
    prediction = str(prediction).strip().lower()

    if prediction == ground_truth:
        return "perfect"
    if "i don't know" in prediction or "i do not know" in prediction:
        return "missing"
    if CRAG_evaluation(item["query"], ground_truth, prediction) == 1:
        return "acceptable"
    return "incorrect"


def record_result(label, counts, n, total_calls, total_tokens, elapsed):
    crag_score = counts["perfect"] + 0.5 * counts["acceptable"] - counts["incorrect"]
    RESULTS[label] = {
        "label": label,
        **counts,
        "n": n,
        "crag_score": crag_score,
        "crag_score_norm": crag_score / n,
        "avg_calls": total_calls / n,
        "avg_tokens": total_tokens / n,
        "elapsed_sec": elapsed,
    }
    save_results()
    print(f"\n[{label}] CRAG score={crag_score:+.1f} ({crag_score / n:+.3f} per question)"
          f"\n         perfect={counts['perfect']}  acceptable={counts['acceptable']}"
          f"  missing={counts['missing']}  incorrect={counts['incorrect']}"
          f"\n         avg LLM calls/question={total_calls / n:.2f}")
    return RESULTS[label]


def evaluate_system(label, infer_fn, items=None):
    """`infer_fn(item) -> answer string`. Every system is scored through this one function."""
    items = items or eval_items
    counts = {"perfect": 0, "acceptable": 0, "missing": 0, "incorrect": 0}
    total_calls, total_tokens = 0, 0
    started = time.time()

    for item in tqdm(items, desc=label):
        with LLMCallCounter() as c:          # counts inference only, not the judge
            try:
                prediction = infer_fn(item)
            except Exception as e:
                # A crash is not a hallucination. Score it as "missing", not as a wrong answer.
                prediction = "I don't know"
                print(f"  [{label}] inference failed: {type(e).__name__}: {e}")

        total_calls += c.calls
        total_tokens += c.prompt_tokens + c.completion_tokens
        counts[grade_prediction(item, prediction)] += 1

    return record_result(label, counts, len(items), total_calls, total_tokens,
                         time.time() - started)


### 3. L0 and L1: The Baselines

**L0** puts the bare question to the LLM. No retrieval at all. On finance questions this is close to
a worst case: the model has no way to know a stock price, so most of its answers are either refusals
or invented numbers.

**L1** is the very `rag` object from Section I, re-measured on the wider 30-question set so that every
number on the ladder is comparable. `KGQueryEngine` extracts the entities from the question and
`Reader` writes an answer out of whatever came back. There is nowhere that asks whether what came back
was any good.


In [ ]:
### YOUR CODE HERE ###

# L0: no retrieval at all
def infer_L0(item):
    return generate_answer(item["query"])


# L1: the single-pass RAG from Section I - retrieve once, read once
def infer_L1(item):
    return rag.inference(item["query"], item["search_results"], 5)["answer"]


evaluate_system("L0: LLM only", infer_L0)
evaluate_system("L1: single-pass RAG", infer_L1)


### 4. L2: Verification as a Structure

The simplest structure we can add to the ladder is **verification**. After the Reader produces an
answer, we show a verifier the same references alongside the candidate answer and ask one narrow
question: *do these references actually support this answer?*

If the verifier says no, the answer is replaced with `"I don't know"`.

This structure adds no knowledge to the system. It cannot find a fact the retriever missed. All it can
do is stop the system from asserting things the evidence does not back — that is, it learns to
**abstain**. Under the CRAG score that is still a real gain, because each suppressed hallucination
moves from **-1 to 0**. Watch whether the `incorrect` bar shrinks while `perfect` stays flat; that is
all of L2.

L2 can notice that the evidence is insufficient, but it cannot act on that. The next section is about
closing that gap.


In [ ]:
### YOUR CODE HERE ###

VERIFY_PROMPT = """You are a strict verifier.
You are given a Question, the References that were retrieved, and a candidate Answer.
Decide whether the References actually support the candidate Answer.

Rules:
- "supported" is true ONLY if the References contain the facts needed to justify the Answer.
- If the Answer is plausible but the References do not contain it, "supported" is false.
- If the Answer says it does not know, "supported" is false.

Respond with ONLY a JSON object:
{"supported": true or false, "missing_information": "<what is still needed, one short phrase>"}
"""


def json_from_text(text, default=None):
    """Pull the first JSON object out of an LLM response."""
    try:
        return json.loads(text)
    except Exception:
        pass

    decoder = json.JSONDecoder()
    pos = 0
    while True:
        start = text.find("{", pos)
        if start == -1:
            return {} if default is None else default
        try:
            obj, _ = decoder.raw_decode(text[start:])
            return obj
        except ValueError:
            pos = start + 1


def format_references(references, per_chunk=1000, total=4000):
    body = "\n".join(f"- {str(r).strip()[:per_chunk]}" for r in references if str(r).strip())
    return body[:total] if body else "(no references)"


def verify_answer(question, references, answer):
    """Ask whether the references actually support the answer.

    Returns (supported, what_is_missing); the second value is what the loop in L3
    feeds back to the planner as the target for its next search.
    """
    user_message = (
        f"Question: {question}\n\n"
        f"# References\n{format_references(references)}\n\n"
        f"Candidate Answer: {answer}\n"
    )
    raw = oai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": VERIFY_PROMPT},
            {"role": "user", "content": user_message},
        ],
    ).choices[0].message.content

    parsed = json_from_text(raw, {"supported": False, "missing_information": ""})
    return bool(parsed.get("supported", False)), str(parsed.get("missing_information", ""))


# L2: L1 plus a verification step - it can now abstain, but not act on the doubt
def infer_L2(item):
    out = rag.inference(item["query"], item["search_results"], 5)
    supported, _ = verify_answer(item["query"], out["retrieved_results"], out["answer"])
    return out["answer"] if supported else "I don't know"


evaluate_system("L2: RAG + self-verify", infer_L2)


### 5. L3: The Agentic Loop

L2 could notice that the evidence was insufficient and then stop there. L3 closes the loop: when
verification fails, the verifier's complaint goes to a **planner**, which decides what to look for
next and where.

```
                +------------------------------------------------+
                |                                                v
  question  -> PLAN --> ACT (web_search | kg_lookup) --> READ --> VERIFY --> answer
                ^                                                 |
                +--------------- "what is missing" ---------------+
```

Three things make this different from simply trying several times.

1. **The planner routes.** Each turn it chooses between `web_search` (the HTML pages attached to the
   question) and `kg_lookup` (the structured finance Knowledge Graph). This is what Section I's `RAG`
   class could not do: it always called both, in a fixed order, with the original question as the
   query.
2. **The query gets rewritten.** The planner does not re-send the user's question. It writes a new
   query aimed at whatever the verifier said was missing. This is what rescues the failures we saw in
   Task 1, where the document existed but the raw question was a poor retrieval key. Asking the same
   question again would only return the same documents.
3. **It decides when to stop.** Easy questions pass verification on the first turn and end there.
   Only hard ones take a second or a third. The point is that we do not fix the number of turns in
   advance.

And when the turns run out with the verifier still unsatisfied, the loop answers `"I don't know"`
rather than shipping its best guess.


In [ ]:
### YOUR CODE HERE ###

import inspect

PLANNER_PROMPT = """You are the planner of a retrieval agent. Decide the NEXT single action.

Available actions:
- "web_search": search the web pages attached to this question. Good for movies, sports, music,
  encyclopedia facts, news, and anything narrative.
- "kg_lookup": query a structured finance Knowledge Graph. It ONLY covers stock market data:
  price, dividend, P/E ratio, EPS, market capitalization, and general company info.
  Its query should name the company and the metric plainly, e.g. "Microsoft dividend".
- "answer": stop retrieving, the evidence gathered so far is enough.

You are given the Question, the Query Time, the evidence gathered so far, and (if this is not the
first turn) the reason the previous attempt was judged insufficient. Target the missing information:
do not repeat a query that has already been tried.

Respond with ONLY a JSON object:
{"action": "web_search" or "kg_lookup" or "answer",
 "query": "<query string for the chosen tool>",
 "reason": "<one short sentence>"}
"""


class AgenticRAG:
    """
    plan -> act -> read -> verify, repeated until the verifier is satisfied
    or `max_steps` turns have been spent.
    """

    def __init__(self, server=None, max_steps=3, topk=5, verbose=False):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine(server=server)
        self.reader = Reader()
        self.max_steps = max_steps
        self.topk = topk
        self.verbose = verbose
        # Section III redefines Reader with a query_time argument; support both signatures.
        self._reader_takes_time = (
            len(inspect.signature(self.reader.generate_response).parameters) >= 3
        )

    def plan(self, question, query_time, evidence, critique, tried):
        user_message = (
            f"Question: {question}\n"
            f"Query Time: {query_time}\n\n"
            f"# Evidence gathered so far\n{format_references(evidence[-5:], per_chunk=300)}\n\n"
            f"# Queries already tried\n{tried or '(none)'}\n\n"
            f"# Why the previous attempt was insufficient\n{critique or '(first turn)'}\n"
        )
        raw = oai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": PLANNER_PROMPT},
                {"role": "user", "content": user_message},
            ],
        ).choices[0].message.content

        plan = json_from_text(raw, {})
        action = plan.get("action", "web_search")
        if action not in ("web_search", "kg_lookup", "answer"):
            action = "web_search"
        return action, str(plan.get("query") or question), str(plan.get("reason", ""))

    def act(self, action, tool_query, search_results):
        if action == "web_search":
            return list(self.retriever.retrieve(tool_query, search_results, self.topk))
        if action == "kg_lookup":
            kg_results = self.kg_query_engine.query(tool_query)
            return [kg_results] if kg_results else []
        return []

    def read(self, question, query_time, evidence):
        if self._reader_takes_time:
            return self.reader.generate_response(question, query_time, evidence)
        return self.reader.generate_response(question, evidence)

    def inference(self, query, search_results, query_time=None):
        evidence, tried, steps = [], [], []
        critique, answer = None, "I don't know"

        for step in range(self.max_steps):
            action, tool_query, reason = self.plan(query, query_time, evidence, critique, tried)

            # Refusing to retrieve before any evidence exists would just be L0 with extra steps.
            if action == "answer" and not evidence:
                action = "web_search"

            if action != "answer":
                tried.append(f"{action}: {tool_query}")
                evidence.extend(self.act(action, tool_query, search_results))

            answer = self.read(query, query_time, evidence)
            supported, missing = verify_answer(query, evidence, answer)

            steps.append({"step": step, "action": action, "query": tool_query, "reason": reason,
                          "answer": answer, "supported": supported, "missing": missing})
            if self.verbose:
                status = "verified" if supported else f"insufficient ({missing})"
                print(f"  step {step}: {action}('{tool_query[:60]}') -> '{answer[:60]}' [{status}]")

            if supported:
                break
            critique = missing
        else:
            # Out of turns and still unverified: say so instead of guessing.
            answer = "I don't know"
            if self.verbose:
                print("  loop exhausted without verification -> \"I don't know\"")

        return {"answer": answer, "evidence": evidence, "steps": steps}


agentic_rag = AgenticRAG(server=external_kg_server, max_steps=3, topk=5, verbose=True)


Before scoring the whole set, run the loop on a few questions with `verbose=True` and read the trace.
This is the most informative part of the section: you can see which tool the planner picked, how it
rewrote the query after a failed verification, and — on the questions it eventually gives up on — the
moment it decides to answer `"I don't know"` instead of inventing a number.

Compare each trace with what L1 produced for the same question in a single pass.


In [ ]:
### YOUR CODE HERE ###

for item in eval_items[:3]:
    print("=" * 90)
    print(f"Q: {item['query']}")
    print(f"   query time   : {item['query_time']}")
    print(f"   ground truth : {item['answer']}")

    print("\n  [L1] single-pass RAG")
    l1_answer = rag.inference(item["query"], item["search_results"], 5)["answer"]
    print(f"  -> {l1_answer}")

    print("\n  [L3] agentic loop")
    out = agentic_rag.inference(item["query"], item["search_results"], item["query_time"])
    print(f"  -> {out['answer']}   ({len(out['steps'])} turn(s))")
    print()


In [ ]:
### YOUR CODE HERE ###

agentic_rag.verbose = False  # quiet for the scored run


def infer_L3(item):
    return agentic_rag.inference(
        item["query"], item["search_results"], item["query_time"]
    )["answer"]


evaluate_system("L3: agentic loop (router)", infer_L3)


### 6. Reading the Results

Read the panels left to right. The last one is deliberately last.

**Left — how well each system does.** CRAG score per question. Look at how far the bar rises each
time a structure is added. If the step from L0 to L1 is the largest, then retrieval alone did more
work than everything else combined; the L2 and L3 steps tell you what was gained after that.

**Middle — why it turned out that way.** Which of the four grades each answer landed in. Watch the
red `incorrect` band. Most of L2's gain, and a good part of L3's, comes from that band shrinking into
grey `missing` rather than turning green. The moment a verifier is attached the system learns to
abstain, and under the CRAG score abstention is `0`, not `-1`. A system that knows what it does not
know earns real points here, and in production it is worth considerably more than this chart suggests.

**Right — and how often does it call the LLM?** Holding the cost back until now was deliberate. If
the loop raised the score, that was the work of feedback and routing, not of making more calls. Call
a single-pass RAG five times and keep only the first answer: five times the calls, and not one point
of difference.

Still, the loop does call a lot, and somebody pays for it. So now we look. How many times taller is
L3's bar than L1's? And is the height gained from L1 to L3 in the left panel worth paying that
multiple for? This is the question you actually face when you deploy a system.

Notice too that L3's average sits well below its worst case of 12 calls on a single question, because
the loop leaves on the first turn for easy questions. Which means the way to bring the average down is
not to make the loop shallower but to make the verifier more accurate: every answer it fails to pass
when it should have shows up directly as calls.

If L3 is only marginally better than L2, look at the traces rather than the score. The usual causes
are a planner that keeps choosing `web_search` for questions the web cannot answer, or a verifier so
strict that everything ends in `"I don't know"` — a high CRAG score and a useless system.


In [ ]:
### YOUR CODE HERE ###

import matplotlib.pyplot as plt

GRADE_COLORS = {"perfect": "#2f855a", "acceptable": "#68d391",
                "missing": "#cbd5e0", "incorrect": "#e53e3e"}


def plot_results(keys=None, title=None):
    """
    Quality, then the grade breakdown that explains it, and only then the cost.

    Cost is deliberately the last panel rather than an axis: plotting a line through
    (calls, quality) would read as "more calls produce more quality", which is not
    what this experiment shows. Systems keep the order they were passed in - the
    ladder order, not cost order.
    """
    keys = [k for k in (keys or list(RESULTS.keys())) if k in RESULTS]
    rows = [RESULTS[k] for k in keys]
    if not rows:
        print("no results to plot yet")
        return

    names = [r["label"].split(":")[0] for r in rows]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    if title:
        fig.suptitle(title)

    def annotate(ax, values, fmt):
        for x, v in zip(names, values):
            ax.annotate(fmt.format(v), (x, v), ha="center",
                        va="bottom" if v >= 0 else "top",
                        xytext=(0, 4 if v >= 0 else -4),
                        textcoords="offset points", fontsize=9)

    # 1. how well each system does
    ax = axes[0]
    scores = [r["crag_score_norm"] for r in rows]
    ax.bar(names, scores, color=["#2b6cb0" if s >= 0 else "#e53e3e" for s in scores])
    ax.axhline(0, color="gray", linewidth=1)
    ax.set_ylabel("CRAG score per question")
    ax.set_title("Quality: how well each system does")
    annotate(ax, scores, "{:+.2f}")

    # 2. why the quality bars move
    ax = axes[1]
    bottom = np.zeros(len(rows))
    for grade, color in GRADE_COLORS.items():
        values = np.array([r[grade] for r in rows], dtype=float)
        ax.bar(names, values, bottom=bottom, label=grade, color=color)
        bottom += values
    ax.set_ylabel("# of questions")
    ax.set_title("Why: where the answers land")
    ax.legend(loc="upper right", fontsize=8)

    # 3. and only now, what it costs
    ax = axes[2]
    calls = [r["avg_calls"] for r in rows]
    ax.bar(names, calls, color="#a0aec0")
    ax.set_ylabel("Average LLM calls per question")
    ax.set_title("Cost: how often it calls the LLM")
    annotate(ax, calls, "{:.1f}")

    plt.tight_layout()
    plt.show()

    print(f"{'system':<32}{'CRAG/q':>9}{'calls/q':>9}{'tokens/q':>11}")
    for r in rows:
        print(f"{r['label']:<32}{r['crag_score_norm']:>9.3f}{r['avg_calls']:>9.2f}"
              f"{r['avg_tokens']:>11.0f}")


plot_results(["L0: LLM only", "L1: single-pass RAG",
              "L2: RAG + self-verify", "L3: agentic loop (router)"])


### 7. What This Experiment Cannot Show

**L3 scoring above L2 does not by itself mean the loop structure is responsible.** L3 differs from L2
in structure, but also in call budget. To tell whether the gain came from feedback or merely from
trying several times, you need a control that spends the same budget without the structure. Running a
single-pass RAG five times and taking a majority vote — self-consistency — is such a control. Only
when that control scores below L3 can you say *"it was the loop, not the calls."*

**Every added call is another opportunity for an error to compound.** A planner that misreads the
question sends a bad query, bad evidence comes back, and the verifier may well pass it. Adding
structure is not always a gain. That is exactly why we measured instead of assuming.

**And the router only chose where to look.** The planner picks between the web and the KG, but
everything that happens after it picks `kg_lookup` is what we hand-wrote in Task 2: an LLM extracts
entities from the question into JSON, and that JSON goes into a fixed chain of
`finance_get_company_name` -> `finance_get_ticker_by_name` -> a metric lookup. The model decides where
to look; how to ask is still whatever we hardcoded.

We have never checked whether that layer holds up. Section III.1 does exactly that — does the same
question produce the same query, and is what comes back a usable size? Then Section III.2 removes the
layer altogether: instead of extracting entities into a decision tree, the server advertises its tools
and the model calls `finance_get_eps(ticker)` itself. As a bonus, adding a tool no longer means
editing the agent.


## III. Upgrade KG Query Stage with MCP

So far, we have integrated RAG with a knowledge graph, enabling the LLM to retrieve structured knowledge through KGQueryEngine. In this approach, the retrieval was based on a decision tree to invoke API calls, and queries were constructed using the LLM.

However, is our KGQueryEngine truly reliable in practical settings? Can the retrieved results be used effectively by the LLM? More importantly, how does this method compare to the emerging MCP-based retrieval approach?

To answer these questions, we will upgrade our RAG system and evaluate both methods through the following steps:

####1. Error Case Analysis
####2. Implement MCP-based Tool Calls
####3. Measuring the MCP Agent with the Same Ruler

This structured comparison will help us determine whether the shift toward MCP tools—now gaining popularity in LLM applications—is practically justified.

###1. Error Case Analysis

Previously, we built Graph RAG and confirmed that incorporating a knowledge graph, rather than relying solely on RAG, can improve performance for certain examples.

However, it remains uncertain whether our `KGQueryEngine` functions correctly for all question-answer pairs. We have yet to analyze all questions within the finance domain.

Beyond questions that require information such as EPS, there can be various other questions within the finance domain. For example, consider the following questions:

In [ ]:
### YOUR CODE HERE ###

with bz2.open(file_path, 'rt') as file:
  for line in file:
    data = json.loads(line.strip())

    if data['interaction_id'] == "7a77679e-d88b-4acf-9532-94e32233950b":
      question = data['query']
      gold_answer = data['answer']
      search_results = data['search_results']
      query_time = data['query_time']
      break

print("Question: ", question)
print("Gold Answer: ", gold_answer)
print("Query Time: ", query_time)

Let’s examine the results generated by our KGQueryEngine for this question.

To do so, we need to recall how we perform retrievals on the knowledge graph. We use an LLM to generate queries that serve as inputs for APIs connected to the knowledge graph.

However, the LLM models provided by OpenAI inherently exhibit randomness, meaning that the same query is not always generated consistently. As a result, our KGQueryEngine does not always return identical results.

Therefore, it is important to repeat the same process multiple times to identify trends in the generated outputs.

Let’s explore the irregularity of GPT and our query generation examples through the following case study.


In [ ]:
### YOUR CODE HERE ###

kg_query_engine = KGQueryEngine(server=external_kg_server)

generated_queries = []

for i in range(3):
    generated_query = kg_query_engine.generate_query(question)[0]
    print(generated_query)
    generated_queries.append(generated_query)

Select queries that explicitly specify company names, metrics, and other relevant details, and examine the results of knowledge graph retrieval


In [ ]:
### YOUR CODE HERE ###

i=0 #change this index

kg_results = kg_query_engine.get_finance_kg_results(generated_queries[i])

json_strings = kg_results.split("<DOC>\n")

json_strings = [s.replace("'", '"') for s in json_strings]

parsed_json = [json.loads(js) for js in json_strings]

for idx, data in enumerate(parsed_json):
    print(json.dumps(data, indent=4))

len(kg_results)

Upon reviewing the search results, we observed that an extremely long string was retrieved.  

Such lengthy search results can lead to the following issues:  

1. **Excessive unnecessary information may cause hallucinations.**  
2. **Even if the necessary information is retrieved, the LLM may fail to recognize it properly.**  
3. **When combined with search results from `search_results`, the total context length may exceed the LLM’s limit, leading to inference errors.**  

Let’s analyze how our RAG responds to this issue. Here, we will focus on handling search results from the knowledge graph, excluding the search process from `search_results`.  

Therefore, let’s declare a new RAG class as follows and use it for evaluation.

In [ ]:
### YOUR CODE HERE ###

class RAGwithoutSR:
    def __init__(self, server=None):
        self.retriever = LlamaIndexRetriever()
        self.kg_query_engine = KGQueryEngine(server=server)
        self.reader = Reader()

    def retrieve(self, query, search_results, topk):
        kg_results = self.kg_query_engine.query(query)

        return [kg_results]

    def generate_response(self, query, retrieved_results):
        answer = self.reader.generate_response(query, retrieved_results)
        return answer

    def inference(self, query, search_results, topk):
        retrieved_results = self.retrieve(query, search_results, topk)
        answer = self.generate_response(query, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }

rag_kg = RAGwithoutSR(server=external_kg_server)

In [ ]:
### YOUR CODE HERE ###

for i in range(3):
    rag_output = rag_kg.inference(question, search_results, 5)
    print(rag_output['answer'])

###2. Implement MCP-based Tool Calls

In this step, we replace the existing KGQueryEngine logic with MCP-based tool calls.

**Model Context Protocol (MCP)** is a standardized interface that allows large language models to interact with external tools or data services via a structured client-server protocol. Each function on the server is registered as a tool, and the LLM can invoke these tools by sending structured requests through the MCP client.

<img src="https://i.imgur.com/P1g0TPh.png">

Therefore, if an external data source can be accessed by the LLM through MCP, this interaction can be viewed as a form of RAG. This is especially relevant in our scenario, where the knowledge graph is only partially accessible via APIs. In such cases, instead of relying on a pre-defined decision tree, it might be more effective to let the LLM decide which tools to use dynamically.

In other words, the LLM itself selects the appropriate tool to retrieve specific information from the knowledge graph.

To enable this, it is crucial to define which tools are available to the LLM. This is handled on the MCP server side, where each tool is registered along with a description that helps the model understand what it can do.

<img src="https://i.imgur.com/EiQfJxQ.png" width=500>

The MCP server consists of core tool definitions and server-side settings that together define the server’s behavior.  Thus, we exclude server execution from this exercise, but you can refer to the attached code for implementation details.

We can leverage the same tools we used previously, such as llamaindex, to construct the MCP client. Refer to the code below to see how the MCP client can be implemented.

Now, let’s proceed to implement the MCP client. This client is responsible for retrieving tool information from the server and forwarding tool invocation requests on behalf of the LLM.

First, let's import what to need.

In [ ]:
### YOUR CODE HERE ###

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult, ToolCall
from llama_index.core.workflow import Context

import dotenv

Second, apart from this, the current reader is not considering query_time. Therefore, let's use the revised version to consider query_time.

In [ ]:
### YOUR CODE HERE ###

from openai import OpenAI

oai_client = OpenAI()

# Maximum characters of reference text passed to the Reader.
# import_function.py's Reader applies the same limit, so every system on the ladder
# in Section II reads a prompt of comparable size.
MAX_CONTEXT_REFERENCES_LENGTH = 4000

class Reader:
  def __init__(self):

    self.system_prompt = """
    You are provided with a question, the time at which the question is asked, and various references.
    Your task is to answer the question succinctly, using the fewest words possible.
    If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
    There is no need to explain the reasoning behind your answers.
    """

  def generate_response(self, question: str, query_time: str, top_k_chunks: list) -> str:
    """
    Generate answer from context.
    """
    llm_input = self.prompt_generator(question, query_time, top_k_chunks)
    completion = oai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=
    llm_input
    ).choices[0].message.content
    return completion

  def prompt_generator(self, query, query_time, top_k_chunks):
    user_message = ""
    references = ""

    if len(top_k_chunks) > 0:
        references += "# References \n"
        # Format the top sentences as references in the model's prompt template.
        for chunk_id, chunk in enumerate(top_k_chunks):
            references += f"- {chunk.strip()}\n"

    references = references[:MAX_CONTEXT_REFERENCES_LENGTH]
    # Limit the length of references to fit the model's input size.

    user_message += f"{references}\n------\n\n"
    user_message
    user_message += f"Using only the references listed above, answer the following question: \n"
    user_message += f"Question: {query}\n"
    user_message += f"Query Time: {query_time}\n"

    llm_input = [
    {"role": "system", "content": self.system_prompt},
    {"role": "user", "content": user_message},
    ]

    return llm_input

### MCP Server Overview

The **MCP (Model-Context-Program) Server** serves as the execution and data layer that interfaces directly with the LLM. It exposes customized functionality and structured contextual information through three main elements: **Resources**, **Tools**, and **Prompts**. This enables LLMs to make informed, context-aware decisions and perform real-world operations safely and efficiently.

---

### MCP Context Components
---

### 1. Resource

A **Resource** provides structured, external data that the LLM can reference during reasoning or decision-making. It represents external objects—such as datasets, files, graph nodes, or static configuration tables—that are read-only from the model's perspective.

**Key Properties:**
- `name`: Human-readable name shown to users or in debugging.
- `description`: A short summary explaining the resource's purpose.
- `mime_type`: The MIME type of the content (e.g., `text/plain`, `application/json`). This describes the content format.
- `uri`: A globally unique identifier for the resource.
- `content`: The actual data or a pointer to the data (e.g., a JSON object, string, or file reference).

**Usage Notes:**
- Resources are **not invoked** like functions.
- They are **fetched by URI**, and remain **static** so that the LLM can repeatedly refer to them without reloading.
- Resources improve LLM accuracy by anchoring it to reliable external context.

---

### 2. Tool

A **Tool** is a callable function registered on the MCP server that the LLM can use to interact with the outside world. These tools bridge natural language instructions and real-world effects by turning a model’s intent into executable actions.

**Typical Use Cases:**
- Executing database queries
- Fetching live data from APIs
- Performing calculations or summarizations
- Searching documents or files

**How It Works:**
1. The user provides a natural language input (e.g., “Please calculate 3 + 7”).
2. The MCP client supplies the LLM with a list of available tools and their schema.
3. The LLM chooses the appropriate tool and extracts input arguments (e.g., `x=3`, `y=7`).
4. The tool invocation is generated and passed to the MCP server.
5. The tool is executed on the server, and the result is returned.
6. The LLM incorporates the result into its context to generate a final user response.

**Implementation:**
Tools are registered using the `@mcp.tool()` decorator in Python, and must conform to the function signature and type schema required by the MCP framework.

---

### 3. Prompt

A **Prompt** in MCP serves as a reusable, templated instruction that guides the LLM’s reasoning in a structured way. Prompts help the model:

- Interpret ambiguous user queries
- Structure downstream tool invocations
- Extract relevant content from complex input

**Prompt Usage Scenarios:**
- Generate structured summaries from unstructured context
- Classify user intent
- Disambiguate time references like “this year” or “last quarter”

Prompts act as templates or procedural guides that the LLM fills in dynamically based on user input and context. While not executable like tools, they play a key role in shaping how the LLM prepares inputs and interprets outputs.

---

### Summary

| Component | Purpose |
|----------|---------|
| `Resource` | Static external data that LLM can reference |
| `Tool`     | Callable function to perform real-world actions |
| `Prompt`   | Structured, reusable guide for reasoning or template generation |

Together, these elements form the foundation of the **MCP Server**, enabling LLMs to operate safely, reliably, and contextually in real applications.

### MCP Client Overview

In this section, we focus on the **MCP Client**, which serves as the interface between the LLM runtime (e.g., LlamaIndex) and an external **MCP Server**.

### What is the MCP Client?

The MCP Client is responsible for:

- Connecting to the MCP Server over HTTP or SSE.
- Fetching available **tools**, **resources**, and **prompts** exposed by the server.
- Translating natural language requests into structured tool invocations.
- Managing context and integrating responses from the server back into the LLM workflow.

It essentially acts as a **bridge** that allows the language model to interact with external APIs and structured context in a modular and dynamic way.

In [ ]:
### YOUR CODE HERE ###

external_mcp_server = "http://10.2.0.165:8081/sse" #change to correct uri

mcp_client = BasicMCPClient(external_mcp_server)
mcp_tool = McpToolSpec(client=mcp_client)

tools = await mcp_tool.to_tool_list_async()
print("\n=== Available Tools ===\n")
for tool in tools:
    print(f"🔧 Name       : {tool.metadata.name}")
    print(f"   Description: {tool.metadata.description}\n")

In [ ]:
resources = await mcp_tool.fetch_resources()

print("\n=== Available Resources ===\n")
for resource in resources:
    print(f"📦 URI        : {resource.uri}")
    print(f"   Name       : {resource.name}")
    print(f"   Description: {resource.description}")
    print(f"   MIME Type  : {resource.mime_type}\n")

#### 1. KG Query Engine Initialization with MCP Client

In this step, we initialize an **LLM agent** that can interact with tools registered on the MCP Server.

- `BasicMCPClient` connects to the MCP Server at the specified URL.
- `McpToolSpec` wraps available tools for the agent to use.
- `FunctionAgent` is created with:
  - The `GPT` model
  - A system prompt guiding tool-based reasoning
  - A dynamic list of MCP tools

The agent is now ready to handle user queries by calling tools exposed via the MCP interface.


In [ ]:
### YOUR CODE HERE ###

SYSTEM_PROMPT = """\
You are an AI assistant for Tool Calling.

Before you help a user, you need to work with tools to interact with Our Knowledge Graph
"""

In [ ]:
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class KGQueryEngineWithMCP:
    def __init__(self, mcp_tool_spec: McpToolSpec, model: str, llm = None):
        self.llm = llm or OpenAI(model=model, temperature=0)
        self.mcp_tool_spec = mcp_tool_spec
        self.agent: Optional[FunctionAgent] = None
        self.agent_context: Optional[Context] = None

    async def init_agent(self):
        tools = await self.mcp_tool_spec.to_tool_list_async()
        for tool in tools:
            if len(tool.metadata.description) > 1000:
                tool.metadata.description = tool.metadata.description[:1000] + "..."
        self.agent = FunctionAgent(
            name="Agent",
            description="An agent that can work with Our Knowledge Graph api.",
            tools=tools,
            llm=self.llm,
            system_prompt=SYSTEM_PROMPT,
        )
        self.agent_context = Context(self.agent)

    async def query(self, question: str, verbose: bool = False) -> str:
        if self.agent is None or self.agent_context is None:
            raise RuntimeError("Agent not initialized. Call `await init_agent()` first.")

        handler = self.agent.run(question, ctx=self.agent_context)
        async for event in handler.stream_events():
            if verbose and type(event) == ToolCall:
                print(f"Calling tool {event.tool_name} with kwargs {event.tool_kwargs}")
            elif verbose and type(event) == ToolCallResult:
                print(f"Tool {event.tool_name} returned {event.tool_output}")

        response = await handler
        return str(response)

kg_engine = KGQueryEngineWithMCP(mcp_tool, model='gpt-3.5-turbo')
await kg_engine.init_agent()

This function executes an LLM agent `FunctionAgent` using a given user message and shared workflow context `Context`.    
It streams intermediate tool call events in real time and returns the final response from the agent.

1. Starts the agent with the input message and context.
2. Streams events while the agent is running.
   - Logs tool invocations and results if `verbose=True`.
3. Awaits the final result and returns it as a string.

This function helps monitor the reasoning and tool execution steps taken by the agent in a transparent, asynchronous manner.

Through the code below, let's see how the MCP-based RAG is solving the 'error case seen above

In [ ]:
### YOUR CODE HERE ###

print("Question: ", question)

response = await kg_engine.query(
    question,
    verbose=True,
)
print(response)

Let's examine the result of the code above.  
Did the MCP agent generate a correct answer? Probably not.

The reason becomes clear when we look at the question that was passed to the LLM.  
The LLM did not receive any information about the **query time**, which is crucial for answering this type of question.  
Since the timing context is highly important here, let’s include the query time and try the question again.

In [ ]:
### YOUR CODE HERE ###

question_with_time = "Do not infer the time scope from the query; use the query time as the literal current moment, and treat any relative expression as anchored to that exact date\n" + "Query: " + question + "\nQuery time: " + query_time
print("Question: ", question_with_time)

response = await kg_engine.query(
    question_with_time,
    verbose=True,
)
print(response)

Was the generated result good enough?

Some users may find it insufficient, while others may encounter outright errors.  
In most cases, these issues arise because the LLM used in MCP exceeded its **context limit** due to the large amount of input.

There are several possible solutions to this problem, but one practical approach is to switch to a model with a **larger context window**.

This time, let's proceed by using the `gpt-4o-mini` model, which supports a larger input context and is better suited for handling longer queries.

In [ ]:
### YOUR CODE HERE ###

kg_engine = KGQueryEngineWithMCP(mcp_tool, model='gpt-4o-mini')
await kg_engine.init_agent()

In [ ]:
### YOUR CODE HERE ###

print("Question: ", question_with_time)

response = await kg_engine.query(
    question_with_time + "\nWhen inferring time information, rely solely on the query time. Do not infer the time scope from the query itself.",
    verbose=True,
)
print(response)

### 2. Structuring a RAG Class for MCP-based Interaction

To modularize the code above and enable scalable, query-driven interaction with the MCP Server, we can refactor it into a unified `RAG` class.

### Design Motivation

- MCP-based clients require **rich, well-structured input** to maximize tool usage accuracy.
- The CRAG dataset provides us with structured components such as:
  - Query (question)
  - Context (retrieved passages)
  - Metadata (timestamps, entities, etc.)

By encapsulating this into a class, we can:
1. **Generate precise questions** from structured CRAG input
2. **Run those questions through the MCP-connected agent**
3. **Return clean, tool-integrated answers**

---

### Key Components

| Method | Purpose |
|--------|---------|
| `__init__` | Initialize MCP client, tools, and LLM agent |
| `retrieve(query, query_time)` | Convert CRAG sample into a natural-language query with max context and retrieve from knowledge graph |
| `generate_response(query, query_time, )` | Perform LLM reference based on searched results to obtain LLM response |
| `inference(query, query_time)` | Combine the above two methods into a method that can perform QATask in RAG |

In [ ]:
### YOUR CODE HERE ###

from llama_index.llms.openai import OpenAI

class RAGwithMCP:
    def __init__(self, mcp_tool):
        self.mcp_application = KGQueryEngineWithMCP(mcp_tool, model='gpt-4o-mini')
        self.reader = Reader()

    async def retrieve(self, query: str, query_time: str, search_results: list, topk: int):
        await self.mcp_application.init_agent()

        full_query = f"""Query: {query} When inferring time information, rely solely on the query time. Do not infer the time scope from the query itself.
Query time: {query_time}"""

        mcp_result = await self.mcp_application.query(full_query, verbose=False)
        return mcp_result

    def generate_response(self, query: str, query_time: str, retrieved_results):
        # Reader iterates over its chunks, so a bare string would be split character by
        # character and the references would arrive as '- M', '- i', '- c', ...
        references = (retrieved_results if isinstance(retrieved_results, list)
                      else [retrieved_results])
        answer = self.reader.generate_response(query, query_time, references)
        return answer

    async def inference(self, query: str, search_results: list, query_time: str, topk: int):
        retrieved_results = await self.retrieve(query, query_time, search_results, topk)
        answer = self.generate_response(query, query_time, retrieved_results)
        return {
            "retrieved_results": retrieved_results,
            "answer": answer
        }

rag_mcp = RAGwithMCP(mcp_tool)

In [ ]:
### YOUR CODE HERE ###

result = await rag_mcp.inference(
    query=question,
    search_results=[],
    query_time=query_time,
    topk=5
)

print("Retrieved Results:\n", result["retrieved_results"])
print("------------------")
print("Final Answer:\n", result["answer"])

In [ ]:
### YOUR CODE HERE ###

for i in range(3):
    rag_output = await rag_mcp.inference(
        query=question,
        search_results=[],
        query_time=query_time,
        topk=5
    )
    print(rag_output['answer'].lower())

In this notebook, we explored how MCP (Model Context Protocol) can be used to build a flexible and modular tool-augmented RAG pipeline.

While MCP allows LLMs to dynamically select and invoke tools using a standardized interface, real-world usage has revealed several practical limitations:

- **Context Overflow**: Tool outputs are inserted directly into the model's input context. If the result is too long, it may exceed the model’s token limit and lead to failure.
- **Limited Error Recovery**: When tool execution fails or exceeds limits, LLMs often cannot recover or retry unless explicitly guided to do so.
- **Debugging Difficulty**: Since tool selection and reasoning are tightly coupled inside the model, it is difficult to trace what went wrong without detailed logs or event streaming.
- **Latency and Reliability**: Each tool call requires a round-trip to the server. In multi-step workflows, this can introduce significant delay and failure points.
- **Loss of Developer Control**: Tool behavior is driven by the LLM’s interpretation of the prompt and available tools, making behavior harder to predict or constrain.

Understanding these trade-offs is key to using MCP effectively in real-world LLM applications.